# 02 · Refuerzo de la clase miedo

**Proyecto:** EmoGol — Trabajo de grado, Universidad del Cauca  
**Autor:** Santiago Fernando Ortega Segura · **Director:** Ph.D. Oscar Mauricio Caicedo Rendón

La clase miedo casi no aparecía en la muestra manual. Este notebook busca candidatos con señales léxicas de miedo, agrega 40 de ellos a la muestra de etiquetado manual y actualiza el pool (106.923 comentarios).

*Nombre durante el desarrollo: `01b_Enriquecimiento_Miedo` (algunas salidas conservan esa numeración).*

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re

SEED = 42
np.random.seed(SEED)

RUTA_DATA = '/content/drive/MyDrive/RUTA/A/data_prep'

df_pool = pd.read_csv(f'{RUTA_DATA}/pool_comentarios_limpios.csv')
df_gold = pd.read_csv(f'{RUTA_DATA}/gold_set_para_anotar.csv', encoding='utf-8', encoding_errors='replace')

print(f"Pool: {len(df_pool):,} comentarios")
print(f"Gold set cargado: {len(df_gold):,} filas")

etiquetas_vacias = df_gold['label'].isna().sum()
if etiquetas_vacias > 0:
    print(f"\n⚠️ {etiquetas_vacias} de {len(df_gold)} filas tienen 'label' vacío/NaN.")
    print("Esto normalmente significa que el gold_set_para_anotar.csv en esta carpeta de Drive")
    print("todavía es la plantilla sin etiquetar (o una versión vieja), no tu versión corregida.")
    print("Sube tu CSV ya corregido a esta ruta de Drive y vuelve a correr esta celda antes de seguir.")
else:
    n_miedo_actual = (df_gold['label'].astype(str).str.lower() == 'miedo').sum()
    print(f"Sin etiquetas vacías. miedo actual: {n_miedo_actual}")
    print(df_gold['label'].astype(str).str.lower().value_counts())

Mounted at /content/drive
Pool: 106,963 comentarios
Gold set cargado: 600 filas
Sin etiquetas vacías. miedo actual: 4
label
alegria     165
burla       149
neutral     118
enojo        99
sorpresa     37
tristeza     28
miedo         4
Name: count, dtype: int64


## 1. Señales léxicas de miedo

In [2]:
SENIALES_MIEDO = [
    r"\bmiedo\b", r"\bsusto\b", r"\bnervios", r"\bpanico\b", r"\bp\u00e1nico\b",
    r"\bno aguanto", r"\bse (nos |les )?(puede |puede )?escapar", r"\bcuidado\b",
    r"\btemo\b", r"\btemer", r"\bpreocupa", r"\baterra", r"\bangustia",
    r"\bqu\u00e9 nervios", r"\bda mala espina", r"\bmalo esto\b", r"\bse puede complicar",
]

patron = re.compile("|".join(SENIALES_MIEDO), flags=re.IGNORECASE)

df_pool["posible_miedo"] = df_pool["text"].astype(str).str.contains(patron, na=False)
candidatos = df_pool[df_pool["posible_miedo"]].copy()

candidatos = candidatos[~candidatos["comment_id"].isin(df_gold["comment_id"])]

print(f"Candidatos encontrados: {len(candidatos)}")
if len(candidatos) < 20:
    print("Advertencia: pocos candidatos — considera ampliar la lista de señales léxicas.")

/tmp/ipykernel_5256/96078228.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_pool["posible_miedo"] = df_pool["text"].astype(str).str.contains(patron, na=False)


Candidatos encontrados: 578


## 2. Muestra para etiquetar

In [3]:
N_CANDIDATOS = 40

muestra = candidatos.sample(min(N_CANDIDATOS, len(candidatos)), random_state=SEED).copy()
columnas_finales = ['comment_id', 'video_id', 'created_at', 'text', 'text_length']
muestra[columnas_finales].assign(label='').to_csv(f'{RUTA_DATA}/candidatos_miedo_para_anotar.csv', index=False)

print(f"Guardado: candidatos_miedo_para_anotar.csv ({len(muestra)} filas)")
print("\nEtiqueta cada fila con una de las 7 clases (no todas van a ser miedo, eso es esperado):")
print(['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral'])

Guardado: candidatos_miedo_para_anotar.csv (40 filas)

Etiqueta cada fila con una de las 7 clases (no todas van a ser miedo, eso es esperado):
['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral']


In [5]:
RUTA_CANDIDATOS_ETIQUETADOS = f'{RUTA_DATA}/candidatos_miedo_para_anotar.csv'

df_candidatos_et = pd.read_csv(RUTA_CANDIDATOS_ETIQUETADOS, encoding='utf-8', encoding_errors='replace')
df_candidatos_et['label'] = df_candidatos_et['label'].astype(str).str.strip().str.lower()

EMOCIONES = ['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral']
vacios = df_candidatos_et['label'].isin(['nan', 'none', '']) | df_candidatos_et['label'].isna()
fuera = ~df_candidatos_et['label'].isin(EMOCIONES) & ~vacios

assert vacios.sum() == 0, f"Hay {vacios.sum()} filas sin etiqueta — complétalas antes de continuar."
assert fuera.sum() == 0, "Hay etiquetas fuera de las 7 clases válidas — revísalas."

n_candidatos = len(df_candidatos_et)
n_miedo_nuevo = (df_candidatos_et['label'] == 'miedo').sum()
pct_miedo_nuevo = 100 * n_miedo_nuevo / n_candidatos if n_candidatos else 0

print('Distribución de las nuevas etiquetas:')
print(df_candidatos_et['label'].value_counts())
print(f"\nDe {n_candidatos} candidatos filtrados por palabra clave, {n_miedo_nuevo} resultaron ser miedo real "
      f"({pct_miedo_nuevo:.0f}%) — el resto confirma que el filtro léxico no reemplaza el criterio humano, "
      "solo ayuda a encontrar los casos raros más rápido.")

Distribución de las nuevas etiquetas:
label
miedo       8
sorpresa    7
alegria     7
burla       7
neutral     6
enojo       3
tristeza    2
Name: count, dtype: int64

De 40 candidatos filtrados por palabra clave, 8 resultaron ser miedo real (20%) — el resto confirma que el filtro léxico no reemplaza el criterio humano, solo ayuda a encontrar los casos raros más rápido.


## 3. Unir a la muestra manual y actualizar el pool

In [6]:
columnas_gold = ['comment_id', 'video_id', 'created_at', 'author', 'likes', 'reply_count', 'text', 'text_length', 'label']

for col in columnas_gold:
    if col not in df_candidatos_et.columns:
        df_candidatos_et[col] = np.nan

df_gold_v2 = pd.concat([df_gold[columnas_gold], df_candidatos_et[columnas_gold]], ignore_index=True)
df_gold_v2 = df_gold_v2.drop_duplicates(subset=['comment_id'], keep='first')

df_pool_v2 = df_pool[~df_pool['comment_id'].isin(df_gold_v2['comment_id'])].copy()

df_gold_v2.to_csv(f'{RUTA_DATA}/gold_set_para_anotar.csv', index=False)
df_pool_v2.drop(columns=['posible_miedo'], errors='ignore').to_csv(f'{RUTA_DATA}/pool_comentarios_limpios.csv', index=False)

print(f"Gold set final: {len(df_gold_v2)} comentarios")
print(df_gold_v2['label'].str.lower().value_counts())
print(f"\nPool actualizado (sin los que pasaron a gold): {len(df_pool_v2):,} comentarios")

Gold set final: 640 comentarios
label
alegria     172
burla       156
neutral     124
enojo       102
sorpresa     44
tristeza     30
miedo        12
Name: count, dtype: int64

Pool actualizado (sin los que pasaron a gold): 106,923 comentarios
